In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
data=pd.read_csv("CKD.csv")

In [3]:
data.head()

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,2.0,76.459948,c,3.0,0.0,normal,abnormal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,yes,no,yes
1,3.0,76.459948,c,2.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,34.000000,12300.000000,4.705597,no,no,no,yes,poor,no,yes
2,4.0,76.459948,a,1.0,0.0,normal,normal,notpresent,notpresent,99.000000,...,34.000000,8408.191126,4.705597,no,no,no,yes,poor,no,yes
3,5.0,76.459948,d,1.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,poor,yes,yes
4,5.0,50.000000,c,0.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,36.000000,12400.000000,4.705597,no,no,no,yes,poor,no,yes


In [4]:
data['classification'].value_counts()

classification
yes    249
no     150
Name: count, dtype: int64

In [5]:
data=pd.get_dummies(data,drop_first=True)
data=data.astype(int)

In [7]:
data.head()

,age,bp,al,su,bgr,bu,sc,sod,pot,hrmo,...,pc_normal,pcc_present,ba_present,htn_yes,dm_yes,cad_yes,appet_yes,pe_yes,ane_yes,classification_yes
0,2,76,3,0,148,57,3,137,4,12,...,0,0,0,0,0,0,1,1,0,1
1,3,76,2,0,148,22,0,137,4,10,...,1,0,0,0,0,0,1,0,0,1
2,4,76,1,0,99,23,0,138,4,12,...,1,0,0,0,0,0,1,0,0,1
3,5,76,1,0,148,16,0,138,3,8,...,1,0,0,0,0,0,1,0,1,1
4,5,50,0,0,148,25,0,137,4,11,...,1,0,0,0,0,0,1,0,0,1


In [8]:
indep=data.iloc[:,0:27].values

In [12]:
dep=data['classification_yes']

In [13]:
indep

array([[ 2, 76,  3, ...,  1,  1,  0],
       [ 3, 76,  2, ...,  1,  0,  0],
       [ 4, 76,  1, ...,  1,  0,  0],
       ...,
       [51, 70,  3, ...,  0,  0,  0],
       [51, 90,  0, ...,  1,  0,  1],
       [51, 80,  0, ...,  1,  0,  0]])

In [14]:
dep

0      1
1      1
2      1
3      1
4      1
      ..
394    1
395    1
396    1
397    1
398    0
Name: classification_yes, Length: 399, dtype: int32

In [15]:
from sklearn.model_selection import train_test_split
x_train ,x_test,y_train,y_test=train_test_split(indep,dep,test_size=0.3,random_state=0)

In [16]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [17]:
from sklearn.svm import SVC

In [19]:
from sklearn.model_selection import GridSearchCV

param_grid = {'kernel':['linear','rbf','poly','sigmoid'],
             'gamma':['auto','scale'],
             'C':[10,100,1000,2000,3000]} 



grid = GridSearchCV(SVC(), param_grid, refit = True, verbose = 3,n_jobs=-1,scoring='f1_weighted') 
   
# fitting the model for grid search 
grid.fit(x_train, y_train) 
 

Fitting 5 folds for each of 40 candidates, totalling 200 fits


GridSearchCV(estimator=SVC(), n_jobs=-1,
             param_grid={'C': [10, 100, 1000, 2000, 3000],
                         'gamma': ['auto', 'scale'],
                         'kernel': ['linear', 'rbf', 'poly', 'sigmoid']},
             scoring='f1_weighted', verbose=3)

In [20]:
# print best parameter after tuning 
#print(grid.best_params_) 
re=grid.cv_results_
#print(re)
grid_predictions = grid.predict(x_test) 
   

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)



# print classification report 
from sklearn.metrics import classification_report
clf_report = classification_report(y_test, grid_predictions)




In [21]:
from sklearn.metrics import f1_score
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)


The f1_macro value for best parameter {'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}: 0.9751481237656352


In [22]:
print("The confusion Matrix:\n",cm)

The confusion Matrix:
 [[45  0]
 [ 3 72]]


In [23]:
print("The report:\n",clf_report)

The report:
               precision    recall  f1-score   support

           0       0.94      1.00      0.97        45
           1       1.00      0.96      0.98        75

    accuracy                           0.97       120
   macro avg       0.97      0.98      0.97       120
weighted avg       0.98      0.97      0.98       120



In [24]:
table=pd.DataFrame.from_dict(re)

In [25]:
table

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_gamma,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.005804,0.003720,0.004085,0.003162,10,auto,linear,"{'C': 10, 'gamma': 'auto', 'kernel': 'linear'}",0.982221,0.946663,0.982221,0.964286,0.981894,0.971457,0.014190,24
1,0.002996,0.002225,0.004209,0.003516,10,auto,rbf,"{'C': 10, 'gamma': 'auto', 'kernel': 'rbf'}",0.982221,1.000000,0.982051,1.000000,1.000000,0.992854,0.008752,1
2,0.004157,0.001791,0.004594,0.003438,10,auto,poly,"{'C': 10, 'gamma': 'auto', 'kernel': 'poly'}",1.000000,0.982051,0.964286,0.982051,0.981894,0.982056,0.011294,17
3,0.002478,0.000723,0.004646,0.002479,10,auto,sigmoid,"{'C': 10, 'gamma': 'auto', 'kernel': 'sigmoid'}",0.982221,1.000000,0.946663,0.964286,1.000000,0.978634,0.020755,19
4,0.002161,0.000908,0.005025,0.002441,10,scale,linear,"{'C': 10, 'gamma': 'scale', 'kernel': 'linear'}",0.982221,0.946663,0.982221,0.964286,0.981894,0.971457,0.014190,24
5,0.003468,0.001375,0.005960,0.000471,10,scale,rbf,"{'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}",0.982221,1.000000,0.982051,1.000000,1.000000,0.992854,0.008752,1
6,0.004914,0.000570,0.002467,0.002031,10,scale,poly,"{'C': 10, 'gamma': 'scale', 'kernel': 'poly'}",1.000000,0.982051,0.964286,0.982051,0.981894,0.982056,0.011294,17
7,0.003117,0.003306,0.005923,0.002724,10,scale,sigmoid,"{'C': 10, 'gamma': 'scale', 'kernel': 'sigmoid'}",0.982221,1.000000,0.946663,0.964286,1.000000,0.978634,0.020755,19
8,0.003600,0.001690,0.004705,0.000854,100,auto,linear,"{'C': 100, 'gamma': 'auto', 'kernel': 'linear'}",0.982221,0.946663,0.982221,0.964286,0.981894,0.971457,0.014190,24
9,0.003392,0.001795,0.003910,0.002199,100,auto,rbf,"{'C': 100, 'gamma': 'auto', 'kernel': 'rbf'}",0.982221,1.000000,0.982051,0.982051,1.000000,0.989265,0.008766,3


In [26]:
import pickle

In [27]:
filename="finalized_model_SVM_GridClassification.sav"

In [28]:
pickle.dump(grid ,open(filename,'wb'))

In [29]:
data.columns

Index(['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv',
       'wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal',
       'pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes',
       'appet_yes', 'pe_yes', 'ane_yes', 'classification_yes'],
      dtype='object')

In [30]:
preinput=sc.transform([[6,82,1,95,17,0,137,3,12,38,7500,5.4,0,0,1,0,0,1,0,0,0,0,0,1,0,0,1]])

In [31]:
loaded_model=pickle.load(open("finalized_model_SVM_GridClassification.sav",'rb'))

In [32]:
result=loaded_model.predict(preinput)

In [33]:
result

array([1])